# BuildFail-SLM: 03 - Master Unified Dataset & Stratified JSONL Generation

This notebook merges the cleaned sources into a balanced canonical schema:
1. **Priority Taxonomy Normalization**: Matches specific, high-value failure classes (Docker, Configuration, Dependency, Test, Code Diff, Auth, Network) before generic fallbacks
2. **Cryptographic Deduplication**: Uses SHA256 fingerprinting on (repository + failure chunk), prioritizing records with real diffs
3. **Master Parquet Generation**: Writes `unified_ci_failure.parquet`, `unified_diagnostics.parquet`, and `unified_repairs.parquet` to `data/03_unified/`
4. **Repository-Aware Data Splits**: Splits datasets with zero repository leakage between train/val/test
5. **Balanced Instruction JSONL Formatting**: Generates high-impact, balanced instruction splits in `data/04_finetuning/`

In [9]:
import os
import json
import hashlib
import yaml
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

CLEAN_DIR = Path("../data/02_source_clean") if Path("../data/02_source_clean").exists() else Path("data/02_source_clean")
UNIFIED_DIR = Path("../data/03_unified") if Path("../data/03_unified").exists() else Path("data/03_unified")
FINETUNE_DIR = Path("../data/04_finetuning") if Path("../data/04_finetuning").exists() else Path("data/04_finetuning")
CONFIG_PATH = Path("../configs/taxonomy.yaml") if Path("../configs/taxonomy.yaml").exists() else Path("configs/taxonomy.yaml")

UNIFIED_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    taxonomy_cfg = yaml.safe_load(f)
print("Loaded Taxonomy Categories:", [c["name"] for c in taxonomy_cfg.get("categories", [])])

Loaded Taxonomy Categories: ['Code Error', 'Dependency Error', 'Build Error', 'Test Failure', 'Configuration Error', 'Environment Error', 'Compilation Error', 'Runtime Error', 'Infrastructure Error', 'Authentication / Permission Error', 'Network Error', 'Deployment Error', 'Docker / Container Error', 'Unknown']


## 1. Load All Cleaned Sources & Map via Priority Refined Rules

In [10]:
REFINED_RULES = [
    ('Code Error', ['syntaxerror', 'nameerror', 'typeerror', 'attributeerror', 'indentationerror', 'lint', 'flake8', 'diff']),
    ('Docker / Container Error', ['docker', 'container', 'image management', 'container runtime', 'dockerfile', 'docker build', 'containerd', 'compose']),
    ('Dependency Error', ['dependency', 'dependencies', 'import', 'modulenotfound', 'pip', 'npm', 'yarn', 'maven', 'gem', 'unresolved', 'package', 'node_modules', 'requirements.txt', 'poetry', 'cargo']),
    ('Test Failure', ['test', 'tests', 'assertion', 'pytest', 'junit', 'itestfailure', 'spec', 'failed test', 'tests failed']),
    ('Configuration Error', ['configuration', 'workflow error', 'yaml error', 'malformed yaml', 'config.yml', 'invalid syntax in workflow', 'actions/checkout', 'missing config']),
    ('Authentication / Permission Error', ['auth', 'permission', 'authorization', 'token', 'credential', 'unauthorized', 'forbidden', '403', 'denied', 'security', 'api key']),
    ('Network Error', ['network', 'socket', 'dns', 'econnrefused', 'timed out', 'timeout', 'connection reset', '502', '503', '504', 'unreachable', 'gateway']),
    ('Compilation Error', ['compile', 'compilation', 'javac', 'cannot find symbol', 'gcc', 'clang', 'cargo build', 'typecheck', 'tsc']),
    ('Environment Error', ['environment', 'resource management', 'storage', 'oom', 'out of memory', 'disk space', 'runner failure']),
    ('Runtime Error', ['runtime', 'database', 'exception', 'crash', 'panic', 'segfault', 'nullpointer', 'fatal']),
    ('Infrastructure Error', ['infrastructure', 'ci orchestration', 'runner provisioner', 'agent offline']),
    ('Deployment Error', ['deploy', 'deployment', 'release', 'publish', 'staging', 'production', 'artifactory']),
    ('Build Error', ['build', 'make', 'gradle', 'mvn', 'rake', 'build failed', 'target failed']),
]

def map_to_taxonomy_priority(row):
    if row.get('source_dataset') == 'ci_repair_bench' or row.get('repair_supervision_available'):
        return 'Code Error'
    text = f"{row.get('failure_type', '')} {row.get('failure_chunk', '')}".lower()
    for cat_name, keywords in REFINED_RULES:
        for kw in keywords:
            if kw in text:
                return cat_name
    return 'Build Error'

sources = [
    CLEAN_DIR / "ci_repair_bench/ci_repair_clean.parquet",
    CLEAN_DIR / "cicd_self_healing/cicd_selfhealing_clean.parquet",
    CLEAN_DIR / "logchunks/logchunks_clean.parquet",
    CLEAN_DIR / "java_travis/java_travis_clean.parquet",
    CLEAN_DIR / "ghalogs/ghalogs_clean.parquet",
    CLEAN_DIR / "travis/travis_clean.parquet"
]

dfs = []
for s in sources:
    if s.exists():
        df = pd.read_parquet(s)
        dfs.append(df)
        print(f"Loaded {s.stem}: {len(df)} rows")

if not dfs:
    raise FileNotFoundError(f"No cleaned parquet files found in {CLEAN_DIR}. Please run notebook 02 first.")
df_combined = pd.concat(dfs, ignore_index=True)
print(f"Total combined raw records across 6 sources: {len(df_combined)}")

df_combined["normalized_failure_category"] = df_combined.apply(map_to_taxonomy_priority, axis=1)
print("\nBalanced Category Distribution Across All Sources:")
print(df_combined["normalized_failure_category"].value_counts())

Loaded ci_repair_clean: 565 rows
Loaded cicd_selfhealing_clean: 7777 rows
Loaded logchunks_clean: 797 rows
Loaded java_travis_clean: 123 rows
Loaded ghalogs_clean: 30053 rows
Loaded travis_clean: 20000 rows
Total combined raw records across 6 sources: 59315

Balanced Category Distribution Across All Sources:
normalized_failure_category
Test Failure                         27240
Dependency Error                      8036
Docker / Container Error              7407
Configuration Error                   5815
Build Error                           4396
Authentication / Permission Error     1580
Runtime Error                         1287
Code Error                            1192
Network Error                          877
Deployment Error                       561
Environment Error                      460
Infrastructure Error                   319
Compilation Error                      145
Name: count, dtype: int64


## 2. Cryptographic Deduplication & Saving Unified Parquet Tables

In [11]:
# Compute fingerprint
def compute_fp(row):
    chunk = str(row.get("failure_chunk") or "")[:300].strip().lower()
    repo = str(row.get("repository") or "").strip().lower()
    return hashlib.sha256(f"{repo}|{chunk}".encode("utf-8")).hexdigest()

df_combined["fingerprint"] = df_combined.apply(compute_fp, axis=1)

# Sort so records with real diffs are kept first during deduplication
df_combined = df_combined.sort_values(by="repair_supervision_available", ascending=False)
df_master = df_combined.drop_duplicates(subset=["fingerprint"]).copy()

# Save Unified Tables
df_master.to_parquet(UNIFIED_DIR / "unified_ci_failure.parquet", index=False)

df_diagnostics = df_master[df_master["failure_chunk"].str.len() > 10].copy()
df_diagnostics.to_parquet(UNIFIED_DIR / "unified_diagnostics.parquet", index=False)

df_repairs = df_master[df_master["repair_supervision_available"] == True].copy()
df_repairs.to_parquet(UNIFIED_DIR / "unified_repairs.parquet", index=False)

print(f"Saved unified_ci_failure.parquet:    {len(df_master)} unique records")
print(f"Saved unified_diagnostics.parquet:   {len(df_diagnostics)} diagnostic records")
print(f"Saved unified_repairs.parquet:       {len(df_repairs)} real code-diff repair records")

Saved unified_ci_failure.parquet:    38347 unique records
Saved unified_diagnostics.parquet:   38347 diagnostic records
Saved unified_repairs.parquet:       565 real code-diff repair records


## 3. High-Quality Stratified Instruction Fine-Tuning JSONL Split Generation

In [12]:
def format_instruction(row):
    repo = row.get("repository") or "Unknown/Repository"
    lang = row.get("language") or "Unknown"
    platform = row.get("ci_platform") or "CI/CD Pipeline"
    chunk = str(row.get("failure_chunk") or "").strip()
    category = row.get("normalized_failure_category") or "Unknown"
    root_cause = row.get("root_cause") or f"Failure detected in {lang} CI build."
    diff = str(row.get("actual_diff") or "").strip()
    fix_cmd = str(row.get("fix_command") or "").strip()
    
    prompt_input = (
        f"### CI Platform: {platform}\n"
        f"### Language: {lang}\n"
        f"### Repository: {repo}\n"
        f"### Failure Log Chunk:\n```\n{chunk[:2000]}\n```"
    )
    
    if diff:
        inst = "Analyze the following CI/CD failure log, identify the root cause, classify the failure category according to the 14-class taxonomy, and generate a proposed unified code diff to fix the failure."
        out = (
            f"### Failure Category:\n{category}\n\n"
            f"### Root Cause Analysis:\n{root_cause}\n\n"
            f"### Proposed Code Fix (Unified Diff):\n```diff\n{diff}\n```"
        )
    else:
        inst = "Analyze the following CI/CD failure log, diagnose the root cause, classify the failure category according to the 14-class taxonomy, and suggest remediation steps."
        cmd_sec = f"\n\n### Recommended Fix Command:\n`{fix_cmd}`" if fix_cmd else ""
        out = (
            f"### Failure Category:\n{category}\n\n"
            f"### Root Cause Analysis:\n{root_cause}"
            f"{cmd_sec}"
        )
        
    return {
        "id": str(row.get("record_id")),
        "instruction": inst,
        "input": prompt_input,
        "output": out
    }

# Perform repository-aware train / validation / test splits
gss = GroupShuffleSplit(n_splits=1, train_size=0.85, random_state=42)
train_idx, temp_idx = next(gss.split(df_diagnostics, groups=df_diagnostics["repository"]))

df_train = df_diagnostics.iloc[train_idx]
df_temp = df_diagnostics.iloc[temp_idx]

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.60, random_state=42)
val_idx, test_idx = next(gss_val.split(df_temp, groups=df_temp["repository"]))

df_val = df_temp.iloc[val_idx]
df_test = df_temp.iloc[test_idx]

# Write JSONL splits
for split_df, split_name in [(df_train, "train.jsonl"), (df_val, "validation.jsonl"), (df_test, "test.jsonl")]:
    split_path = FINETUNE_DIR / split_name
    with open(split_path, "w", encoding="utf-8") as f:
        for _, r in split_df.iterrows():
            f.write(json.dumps(format_instruction(r), ensure_ascii=False) + "\n")
    print(f"Saved {split_name}: {len(split_df)} examples")

# Write Isolated OOD Test Set
ood_parquet = CLEAN_DIR / "cicd_self_healing/cicd_ood_clean.parquet"
if ood_parquet.exists():
    df_ood = pd.read_parquet(ood_parquet)
    df_ood["normalized_failure_category"] = df_ood.apply(map_to_taxonomy_priority, axis=1)
    with open(FINETUNE_DIR / "ood_test.jsonl", "w", encoding="utf-8") as f:
        for _, r in df_ood.iterrows():
            f.write(json.dumps(format_instruction(r), ensure_ascii=False) + "\n")
    print(f"Saved ood_test.jsonl (Strictly Isolated OOD Benchmark): {len(df_ood)} examples")

Saved train.jsonl: 32879 examples
Saved validation.jsonl: 3510 examples
Saved test.jsonl: 1958 examples
Saved ood_test.jsonl (Strictly Isolated OOD Benchmark): 489 examples
